# Deep Agents 第 1 阶段：harness 基础

对应 `codes/deepagents/index.md` 第 1 阶段，讲义见同目录 `README.md`。

先运行下一格加载 `DEEPSEEK_API_KEY`（可放 `codes/langgraph/local.env`）。下面 `# 作答` 已补参考实现；有 key 时自己再跑一遍核对即可。


In [ ]:
import os
from pathlib import Path

for p in [Path.cwd(), *Path.cwd().parents]:
    env = p / "codes/langgraph/local.env"
    if env.is_file():
        for line in env.read_text().splitlines():
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
        break

from langchain.chat_models import init_chat_model
model = init_chat_model(f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}")

## 1. 最小 deep agent 与它的工具集

用 `create_deep_agent(model=model)` 建一个代理，让它「把三条 Python 并发要点写进 /notes.md」。

1. 打印 `out.keys()` 和 `out["files"]`。
2. 文件在磁盘上吗？为什么？
3. 工具列表里有没有 `write_todos`？（提示：看 0.7 发布说明）

In [ ]:
from deepagents import create_deep_agent
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage

# 作答
agent = create_deep_agent(model=model)
out = agent.invoke({
    "messages": [{"role": "user", "content": "把三条 Python 并发要点写进 /notes.md"}]
})

print("keys:", sorted(out.keys()))
print("files:", out.get("files"))
print("on disk /notes.md?", Path("/notes.md").exists())
# 默认 StateBackend：文件在图 state 的 files 里，不落磁盘。

# 0.7 起默认没有 write_todos：用假模型看 bind_tools 名单（不耗 API）
class Probe(GenericFakeChatModel):
    seen = []
    def bind_tools(self, tools, **kw):
        self.seen.append(sorted(t.name if hasattr(t, "name") else t["name"] for t in tools))
        return self

probe = Probe(messages=iter([AIMessage("ok")]))
create_deep_agent(model=probe).invoke({"messages": [{"role": "user", "content": "hi"}]})
print("default tools:", probe.seen[-1])
print("write_todos?", "write_todos" in probe.seen[-1])


## 2. 打开计划工具

给第 1 题的代理加上 `TodoListMiddleware`，换一个多步任务（例如「对比 asyncio / threading / multiprocessing，分三节写进 /report.md」）。

1. 用 `agent.stream(..., stream_mode="updates")` 观察 `todos` 的状态变化。
2. 和不加时比，模型调用次数和最终文件有什么差别？

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

# 作答
agent = create_deep_agent(model=model, middleware=[TodoListMiddleware()])

prompt = (
    "对比 asyncio / threading / multiprocessing，分三节写进 /report.md。"
    "先用 write_todos 列出计划，再逐节完成并更新 todos 状态。"
)

print("--- stream updates (todos) ---")
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": prompt}]},
    stream_mode="updates",
):
    for node, delta in chunk.items():
        if isinstance(delta, dict) and delta.get("todos") is not None:
            print(node, "->", delta["todos"])

out = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
print("final todos:", out.get("todos"))
print("files:", list((out.get("files") or {}).keys()))
# 对比第 1 题：多了 todos 通道；通常会多几次 write_todos 调用，最终仍有 /report.md。


## 3. 子代理与上下文隔离

定义一个 `critic` 子代理（`name` / `description` / `system_prompt`），主代理先写草稿，再派 `critic` 审阅。

1. 在 `out["messages"]` 里找到 `task` 对应的 `ToolMessage`，它包含子代理的哪部分输出？
2. 子代理的 `system_prompt` 出现在父代理消息里了吗？
3. 把 `description` 改成一个含糊的词，再跑一次，主代理还会派活吗？

In [ ]:
# 作答
from langchain_core.messages import ToolMessage

critic = {
    "name": "critic",
    "description": "审阅技术草稿：挑漏洞、缺出处、结构问题；只回传审阅结论",
    "system_prompt": "你是苛刻的技术审稿人。只输出：问题列表 + 是否通过。不要复述全文。",
}
agent = create_deep_agent(model=model, subagents=[critic])
out = agent.invoke({
    "messages": [{"role": "user", "content":
        "先把『Python GIL 对线程的影响』写成短草稿到 /draft.md，"
        "再派 critic 子代理审阅，根据审阅意见改一版。"}]
})

task_tools = [
    m for m in out["messages"]
    if isinstance(m, ToolMessage) or getattr(m, "type", None) == "tool"
]
print("tool messages:", len(task_tools))
if task_tools:
    # task 的 ToolMessage 通常只含子代理最终答复，不含中间过程
    print("--- first tool content (prefix) ---")
    print(str(task_tools[0].content)[:600])

parent_blob = "\n".join(str(getattr(m, "content", "")) for m in out["messages"])
print("parent sees critic system_prompt?", "苛刻的技术审稿人" in parent_blob)
# 期望 False：子代理 system_prompt 留在子上下文，不进父 messages。

vague = {
    "name": "critic",
    "description": "助手",  # 含糊：主模型往往不知道何时派
    "system_prompt": "你是审稿人。",
}
out2 = create_deep_agent(model=model, subagents=[vague]).invoke({
    "messages": [{"role": "user", "content": "写短草稿到 /draft.md，必要时可派子代理审阅。"}]
})
n_task = sum(
    1
    for m in out2["messages"]
    for tc in (getattr(m, "tool_calls", None) or [])
    if tc.get("name") == "task"
)
print("vague description → task calls:", n_task)


## 4. 换 backend：长期记忆

用 `CompositeBackend` 把 `/memories/` 路由到 `StoreBackend`（`InMemoryStore`），其余仍走 `StateBackend`。

1. 第一个 `thread_id` 里让代理把「用户偏好：回答简洁」写进 `/memories/pref.md`。
2. 换一个新 `thread_id`，让代理先读这个文件再回答。哪些文件跨线程还在？

注意：用 `checkpointer=MemorySaver()` 才有 thread 概念。

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver

# 作答
store = InMemoryStore()
backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend(namespace=lambda rt: ("user-demo",), store=store)},
)
agent = create_deep_agent(
    model=model,
    backend=backend,
    store=store,
    checkpointer=MemorySaver(),
)

cfg1 = {"configurable": {"thread_id": "t1"}}
out1 = agent.invoke(
    {"messages": [{"role": "user", "content":
        "把『用户偏好：回答简洁』写进 /memories/pref.md；"
        "再写一个临时备忘 /scratch.md。"}]},
    cfg1,
)
print("t1 state files:", list((out1.get("files") or {}).keys()))
print("store items:", store.search(("user-demo",)))

cfg2 = {"configurable": {"thread_id": "t2"}}
out2 = agent.invoke(
    {"messages": [{"role": "user", "content":
        "先读 /memories/pref.md，按偏好用一句话介绍你自己；"
        "再说你能否读到 /scratch.md。"}]},
    cfg2,
)
print("t2 state files:", list((out2.get("files") or {}).keys()))
print(out2["messages"][-1].content)
# /memories/* 在 StoreBackend，跨 thread 仍在；/scratch.md 只活在 t1 的 StateBackend。


## 5. 对比题（答案）

1. **同一种底层，不同一层包装。** `create_deep_agent` 返回的仍是 LangGraph `CompiledStateGraph`，循环形态和预置 ReAct / `create_agent` 一类。差别在 **harness / 中间件层**：默认带上虚拟文件系统、`task` 子代理、摘要、悬空 tool call 修补等，不是另画一张无关的图。

2. **固定三步报表 → 手写 LangGraph（或薄 `create_agent`）**；**调研陌生领域 → `create_deep_agent`**。前者步骤已知、要精确控制边与状态；后者步数未知，需要外存、可选计划、子代理隔离上下文。

3. **`write_todos` 是 no-op：不执行业务，只把计划写入 state 的 `todos`。** 每轮都能看见「做到哪 / 还剩什么」，给注意力一个锚点，减轻长程目标漂移。0.7 起默认关闭（强模型评测提分不明显还多花钱）；弱模型、超长任务、或要给用户展示进度时再 `TodoListMiddleware` opt-in。
